In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import Holt

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Sheet1"
series_col = "y"

df = pd.read_excel(file_path, sheet_name=sheet_name)
y_train = df[series_col].to_numpy(dtype=float)

# ========= 2) 参数模板 =========
params = {
    "forecast_steps": 6,  # int: 预测步数
    "alpha": 0.4,         # float: 水平平滑系数(0,1)
    "beta": 0.2           # float: 趋势平滑系数(0,1)
}

model = Holt(y_train, initialization_method="estimated")
fit = model.fit(
    smoothing_level=params["alpha"],
    smoothing_trend=params["beta"],
    optimized=False
)
y_pred = fit.forecast(params["forecast_steps"])
print(y_pred)


In [ ]:
"""
二次指数平滑预测

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "二次指数平滑预测.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
TARGET_COLUMN = "y"  # TODO: 请填写[目标列名]，说明：单变量时间序列。
ALPHA = 0.4  # TODO: 请填写[水平平滑系数]，说明：0 到 1，越大越重视新数据。
BETA = 0.2  # TODO: 请填写[趋势平滑系数]，说明：0 到 1，越大趋势调整越快。
FORECAST_STEPS = 5  # TODO: 请填写[预测期数]，说明：短期预测更可靠。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    y = data[TARGET_COLUMN].to_numpy(dtype=float)
    level, trend = y[0], y[1] - y[0]
    fitted = []
    for value in y:
        last_level = level
        level = ALPHA * value + (1 - ALPHA) * (level + trend)
        trend = BETA * (level - last_level) + (1 - BETA) * trend
        fitted.append(level + trend)
    forecast = [level + (i + 1) * trend for i in range(FORECAST_STEPS)]
    result = pd.DataFrame({"序号": range(1, len(y) + FORECAST_STEPS + 1), "拟合或预测值": fitted + forecast})
    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(result)


if __name__ == "__main__":
    df = load_data()
    run_model(df)
